# 🧪 ChemSafe Assistant — GNN Training Notebook

This notebook trains a Graph Neural Network (GCN) on the Tox21 dataset to predict 12 toxicity endpoints.

**Designed for Google Colab GPU runtime.**

Pipeline: `Load graphs.pt → Build DataLoaders → Train GNN → Evaluate ROC-AUC → Save Model`

## 1. Setup & Installation
Install required packages on Colab.

In [ ]:
# Run only on Google Colab
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

try:
    import google.colab
    IN_COLAB = True
    print('Running on Google Colab. Installing dependencies...')
    install('torch')
    install('torch-geometric')
    install('rdkit')
    install('scikit-learn')
    print('✅ All packages installed.')
except ImportError:
    IN_COLAB = False
    print('Running locally. Skipping Colab installs.')

In [ ]:
import os
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt

# Mount Google Drive if on Colab
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/chem-safe-assistant'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add project root to path
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Load Processed Data

In [ ]:
GRAPHS_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'graphs.pt')

if not os.path.exists(GRAPHS_PATH):
    print('⚠️ graphs.pt not found. Running preprocessing pipeline first...')
    from src.data.dataset import save_processed_pipeline
    csv_path = os.path.join(PROJECT_ROOT, 'data', 'raw', 'tox21.csv')
    save_processed_pipeline(csv_path, GRAPHS_PATH, desalt=True, seed=42)

data_splits = torch.load(GRAPHS_PATH)
train_graphs = data_splits['train']
val_graphs   = data_splits['val']
test_graphs  = data_splits['test']

print(f'Train: {len(train_graphs)} | Val: {len(val_graphs)} | Test: {len(test_graphs)}')
print(f'Node feature dim: {train_graphs[0].x.shape[1]}')

## 3. Initialize Model

In [ ]:
from src.model.gnn import Tox21GNN

# Get number of node features from the first graph
NUM_NODE_FEATURES = train_graphs[0].x.shape[1]

model = Tox21GNN(
    num_node_features=NUM_NODE_FEATURES,
    hidden_dim=128,
    num_tasks=12,
    dropout=0.3
)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: Tox21GNN')
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(model)

## 4. Train the Model

In [ ]:
from src.model.trainer import Trainer

SAVE_PATH = os.path.join(PROJECT_ROOT, 'saved_models', 'gnn_tox21.pt')

trainer = Trainer(
    model=model,
    train_graphs=train_graphs,
    val_graphs=val_graphs,
    lr=1e-3,
    batch_size=64,
    epochs=100,
    patience=15,
    save_path=SAVE_PATH
)

history = trainer.train()

## 5. Plot Training Metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history['train_losses'], label='Train Loss', color='#2196F3', linewidth=2)
axes[0].plot(history['val_losses'], label='Val Loss', color='#FF5722', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Masked BCE Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ROC-AUC curve
axes[1].plot(history['val_aucs'], label='Val Mean ROC-AUC', color='#4CAF50', linewidth=2)
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Validation ROC-AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📊 Training curves saved.')

## 6. Evaluate on Test Set

In [ ]:
from torch_geometric.loader import DataLoader
from sklearn.metrics import roc_auc_score
from src.data.dataset import TOX21_LABELS

# Load best checkpoint
checkpoint = torch.load(SAVE_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()

test_loader = DataLoader(test_graphs, batch_size=64, shuffle=False)

all_preds = []
all_targets = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        logits = model(batch)
        probs = torch.sigmoid(logits).cpu().numpy()
        labels = batch.y.view(-1, 12).cpu().numpy()
        all_preds.append(probs)
        all_targets.append(labels)

all_preds = np.concatenate(all_preds, axis=0)
all_targets = np.concatenate(all_targets, axis=0)

print('\n📋 Per-Task Test ROC-AUC:')
print('-' * 40)
valid_aucs = []
for i, name in enumerate(TOX21_LABELS):
    y_true = all_targets[:, i]
    y_score = all_preds[:, i]
    mask = ~np.isnan(y_true)
    y_true_v = y_true[mask]
    y_score_v = y_score[mask]
    if len(np.unique(y_true_v)) < 2:
        print(f'  {name:<20} N/A (single class)')
    else:
        auc = roc_auc_score(y_true_v, y_score_v)
        valid_aucs.append(auc)
        print(f'  {name:<20} {auc:.4f}')

mean_auc = np.mean(valid_aucs) if valid_aucs else 0.0
print('-' * 40)
print(f'  {"Mean ROC-AUC":<20} {mean_auc:.4f}')

## 7. Save Final Model Info

In [ ]:
print(f'\n✅ Training complete.')
print(f'Best model checkpoint: {SAVE_PATH}')
print(f'Best val loss: {checkpoint["val_loss"]:.4f}')
print(f'Best val AUC:  {checkpoint["val_auc"]:.4f}')
print(f'Test Mean AUC: {mean_auc:.4f}')